# Cálculo

 Buscando desenvolver a base para cálculo, foram importadas constantes e funções de outros módulos programados anteriormente.

 As operações de cálculo contemplam derivação, a derivação de n-ésima ordem, coeficientes do binômio de Newton, gradiente, hessiano, jacobiano, laplaciano, derivada direcional e série de Taylor; integração definida, com verificação de intervalo, pela regra dos retângulos à esquerda e à direita, pelo ponto médio, pela regra dos trapézios, integral de Simpson e integral de Simpson 3/8 e pelo método de Romberg; integração imprópria; dupla integração; cálculo do comprimento de uma curva; integração de Monte Carlo; derivação de funções e integração indefinida de funções; limites e limites no infinito; e derivadas parciais de primeira e de segunda ordem.

 Foram implementados algoritmos para análises de funções e de seus comportamentos, como para encontrar os zeros de uma função, método da bissecção, mudanças de comportamento como encontrar os extremos, pontos de inflexão. Há algumas funções para checar o comportamento das funções, como monotônia, continuidade, se é par, se é ímpar, se é injetora, sobrejetora ou bijetora. Algoritmos para encontrar a imagem e o domínio aproximados das funções. 

 Também há funções de resolução de equações por ponto fixo, método de Newton-Raphson, método da secante, método da falsa posição. E, por fim, uma função auxíliar para gerar funções com xy.

 Para finalizar, adicionou-se a linha de código "%%writefile calculo.py" para converter o código em um módulo exportável para ser usado em outros módulos e no uso final. Pelo mesmo motivo, o código teve que ser colocado inteiramente em uma única célula de código, pois o comando usado para isso exporta códigos de apenas uma célula para isso.

In [1]:
%%writefile calculo.py

from constantes import tolerancia_de_erro, maximo_de_interacoes
from aritmetica import valor_absoluto, fatorial


#Derivação
def derivada(f, x: float, h: float = 1e-5, metodo: str = "central") -> float:
    if metodo == "progressiva":
        return (f(x + h) - f(x)) / h
    elif metodo == "regressiva":
        return (f(x) - f(x - h)) / h
    elif metodo == "central":
        return (f(x + h) - f(x - h)) / (2.0 * h)
    else:
        raise ValueError(f"Método '{metodo}' desconhecido. Use 'progressiva', 'regressiva' ou 'central'.")


#Derivação de n-ésima ordem
def n_derivada(f, x: float, n: int, h: float = None) -> float:
    if n == 0:
        return f(x)

    if h is None:
        h = tolerancia_de_erro ** (1.0 / (n + 2))
        escala = max(1.0, valor_absoluto(x))
        h *= escala

    if n == 1:
        return derivada(f, x, h, "central")

    binomio = linha_binomial(n)
    meio = n / 2.0
    resultado = 0.0
    for k in range(n + 1):
        sinal = (-1) ** k
        deslocamento = (meio - k) * h
        resultado += sinal * binomio[k] * f(x + deslocamento)

    return resultado / (h ** n)


#Coeficientes do binômio de Newton
def linha_binomial(n: int) -> list:
    linha = [1] * (n + 1)
    for i in range(1, n + 1):
        for j in range(i - 1, 0, -1):
            linha[j] += linha[j - 1]
    return linha


#Gradiente
def gradiente(f, x: list, h: float = 1e-5) -> list:
    n = len(x)
    grad = [0.0] * n
    for i in range(n):
        
        x_mais = x[:]
        x_menos = x[:]
        
        x_mais[i] += h
        x_menos[i] -= h
        
        grad[i] = (f(x_mais) - f(x_menos)) / (2.0 * h)
        
    return grad


#Hessiano
def hessiano(f, x: list, h: float = 1e-4) -> list:
    n = len(x)
    H = [[0.0] * n for _ in range(n)]
    f0 = f(x)

    for i in range(n):
        for j in range(n):
            if i == j:
                
                xp = x[:]
                xm = x[:]
                
                xp[i] += h
                xm[i] -= h
                
                H[i][i] = (f(xp) - 2 * f0 + f(xm)) / (h * h)
                
            elif j > i:
                xpp = x[:]
                xpm = x[:]
                xmp = x[:]
                xmm = x[:]
                
                xpp[i] += h; xpp[j] += h
                xpm[i] += h; xpm[j] -= h
                xmp[i] -= h; xmp[j] += h
                xmm[i] -= h; xmm[j] -= h
                
                H[i][j] = (f(xpp) - f(xpm) - f(xmp) + f(xmm)) / (4 * h * h)
                H[j][i] = H[i][j]

    return H


#Jacobiano
def jacobiano(F, x: list, h: float = 1e-5) -> list:
    n = len(x)
    F_base = F(x)
    m = len(F_base)

    J = [[0.0] * n for _ in range(m)]
    for j in range(n):
        
        x_mais = x[:]
        x_menos = x[:]
        
        x_mais[j] += h
        x_menos[j] -= h
        
        F_mais = F(x_mais)
        F_menos = F(x_menos)
        
        for i in range(m):
            J[i][j] = (F_mais[i] - F_menos[i]) / (2.0 * h)

    return J

#Laplaciano
def laplaciano(f, x: list, h: float = 1e-4) -> float:
    H = hessiano(f, x, h)
    soma = 0.0
    for i in range(len(x)):
        soma += H[i][i]
    return soma


#Derivada direcional
def derivada_direcional(f, x: list, direcao: list, h: float = 1e-5) -> float:
    norma_direcao = sum(d * d for d in direcao) ** 0.5
    if norma_direcao < tolerancia_de_erro:
        raise ZeroDivisionError("Direção não pode ser o vetor nulo.")
    v_unitario = [d / norma_direcao for d in direcao]

    grad = gradiente(f, x, h)
    return sum(grad[i] * v_unitario[i] for i in range(len(x)))


#Coeficientes da série de Taylor
def serie_de_taylor(f, x0: float, ordem: int, h: float = None) -> list:
    coeficientes = []
    for k in range(ordem + 1):
        if k == 0:
            derivada_k = f(x0)
        else:
            derivada_k = n_derivada(f, x0, k, h)
        coeficientes.append(derivada_k / fatorial(k))

    return coeficientes


#Avaliação da série de Taylor
def avaliar_taylor(coeficientes: list, x0: float, x: float) -> float:
    delta = x - x0
    resultado = 0.0
    potencia_delta = 1.0
    for c in coeficientes:
        resultado += c * potencia_delta
        potencia_delta *= delta
    return resultado


#Integração
#Verificacao de intervalo
def verificar_intervalo(a: float, b: float, n: int):
    if a >= b:
        raise ValueError(f"Intervalo inválido: a={a} ≥ b={b}.")
    if n <= 0:
        raise ValueError(f"Número de subdivisões n={n} deve ser > 0.")


#Soma de Riemman
#Retângulos à esquerda
def integral_retangulos_esquerda(f, a: float, b: float, n: int = 1000) -> float:
    verificar_intervalo(a, b, n)
    h = (b - a) / n
    return sum(f(a + i * h) for i in range(n)) * h


#Retângulos à direita
def integral_retangulos_direita(f, a: float, b: float, n: int = 1000) -> float:
    verificar_intervalo(a, b, n)
    h = (b - a) / n
    return sum(f(a + (i + 1) * h) for i in range(n)) * h


#Ponto médio
def integral_ponto_medio(f, a: float, b: float, n: int = 1000) -> float:
    verificar_intervalo(a, b, n)
    h = (b - a) / n
    return sum(f(a + (i + 0.5) * h) for i in range(n)) * h


#Regra dos trapézios
def integral_trapezios(f, a: float, b: float, n: int = 1000) -> float:
    verificar_intervalo(a, b, n)
    h = (b - a) / n
    s = f(a) + f(b)
    for i in range(1, n):
        s += 2.0 * f(a + i * h)
    return s * h / 2.0


#Regra de Simpson padrão (1/3)
def integral_simpson(f, a: float, b: float, n: int = 1000) -> float:
    verificar_intervalo(a, b, n)
    if n % 2 != 0:
        n += 1
    h = (b - a) / n
    s = f(a) + f(b)
    for i in range(1, n):
        coef = 4.0 if i % 2 != 0 else 2.0
        s += coef * f(a + i * h)
    return s * h / 3.0


#Integral de Simpson 3/8
def integral_simpson_3_8(f, a: float, b: float, n: int = 999) -> float:
    verificar_intervalo(a, b, n)
    while n % 3 != 0:
        n += 1
    h = (b - a) / n
    s = f(a) + f(b)
    for i in range(1, n):
        coef = 3.0 if i % 3 != 0 else 2.0
        s += coef * f(a + i * h)
    return s * 3 * h / 8.0


#Método de Romberg
def integral_romberg(f, a: float, b: float, max_k: int = 10, tolerancia: float = None) -> float:
    if tolerancia is None:
        tolerancia = tolerancia_de_erro

    verificar_intervalo(a, b, 1)
    T = [[0.0] * (max_k + 1) for _ in range(max_k + 1)]
    T[0][0] = (f(a) + f(b)) * (b - a) / 2.0

    for k in range(1, max_k + 1):
        n = 2 ** k
        h = (b - a) / n
        pontos_medios = sum(f(a + (2 * i - 1) * h) for i in range(1, n // 2 + 1))
        T[k][0] = T[k - 1][0] / 2.0 + h * pontos_medios

        for j in range(1, k + 1):
            fator = 4.0 ** j
            T[k][j] = (fator * T[k][j - 1] - T[k - 1][j - 1]) / (fator - 1.0)

        if k > 1 and valor_absoluto(T[k][k] - T[k - 1][k - 1]) < tolerancia:
            return T[k][k]

    return T[max_k][max_k]


#Integração imprópria
def integral_impropria(f, a, b, n: int = 10000, tolerancia: float = 1e-8) -> float:
    limiar_infinito = 1e15

    a_infinito = a is None or a <= -limiar_infinito
    b_infinito = b is None or b >= limiar_infinito

    if a_infinito and b_infinito:
        def g(t):
            if valor_absoluto(t) >= 1 - 1e-12:
                return 0.0
            denominador = 1.0 - t * t
            x = t / denominador
            dxdt = (1.0 + t * t) / (denominador * denominador)
            return f(x) * dxdt
        return integral_simpson(g, -1.0 + 1e-8, 1.0 - 1e-8, n)

    elif b_infinito:
        a = float(a)
        def g(t):
            if t >= 1 - 1e-12 or t <= 1e-12:
                return 0.0
            denominador = 1.0 - t
            x = a + t / denominador
            dxdt = 1.0 / (denominador * denominador)
            return f(x) * dxdt
        return integral_simpson(g, 1e-8, 1.0 - 1e-8, n)

    elif a_infinito:
        b = float(b)
        def g(t):
            if t >= 1 - 1e-12 or t <= 1e-12:
                return 0.0
            denominador = 1.0 - t
            x = b - t / denominador
            dxdt = 1.0 / (denominador * denominador)
            return f(x) * dxdt
        return integral_simpson(g, 1e-8, 1.0 - 1e-8, n)

    else:
        return integral_romberg(f, float(a), float(b), tolerancia=tolerancia)


#Dupla integração
def integral_dupla(f, a: float, b: float, g_inferior, g_superior, n: int = 200) -> float:
    def integral_em_y(x):
        y_inferior = g_inferior(x)
        y_superior = g_superior(x)
        if valor_absoluto(y_superior - y_inferior) < tolerancia_de_erro:
            return 0.0
        return integral_simpson(lambda y: f(x, y), y_inferior, y_superior, n)

    return integral_simpson(integral_em_y, a, b, n)


#Cálculo do comprimento de uma curva
def comprimento_de_arco(f, a: float, b: float, n: int = 1000, h: float = 1e-6) -> float:
    def integrando(x):
        d = derivada(f, x, h)
        return (1.0 + d * d) ** 0.5

    return integral_simpson(integrando, a, b, n)



#Integração de Monte Carlo
def integral_monte_carlo(f, a: float, b: float, amostras: int = 100000, semente: int = 42) -> float:
    verificar_intervalo(a, b, amostras)

    estado = semente
    modulo = 2 ** 31 - 1
    multiplicador = 1103515245
    incremento = 12345

    soma = 0.0
    for _ in range(amostras):
        estado = (multiplicador * estado + incremento) % modulo
        u = estado / modulo
        x = a + u * (b - a)
        soma += f(x)

    return (b - a) * soma / amostras


#Derivação de funções
def derivada_de_funcao(expressao_f, ordem: int = 1, h: float = 1e-5):
    if ordem == 1:
        return lambda x: derivada(expressao_f, x, h, "central")
    return lambda x: n_derivada(expressao_f, x, ordem, h)


#Integração indefinida de funções
def integral_indefinida_numerica(f, x0: float = 0.0, h: float = 1e-5, n: int = 1000):
    def F(x):
        if x >= x0:
            return integral_simpson(f, x0, x, max(n, 2))
        return -integral_simpson(f, x, x0, max(n, 2))
    return F


#Cálculo de limites
def limite(f, ponto: float, h: float = 1e-7, lado: str = "ambos") -> float:
    if lado == "esquerda":
        valores = [f(ponto - h * (10 ** -k)) for k in range(1, 10)]
    elif lado == "direita":
        valores = [f(ponto + h * (10 ** -k)) for k in range(1, 10)]
    elif lado == "ambos":
        limite_esquerda = limite(f, ponto, h, "esquerda")
        limite_direita = limite(f, ponto, h, "direita")
        if valor_absoluto(limite_esquerda - limite_direita) > tolerancia_de_erro * 1000:
            raise ValueError("O limite bilateral não existe neste ponto.")
        return (limite_esquerda + limite_direita) / 2.0
    else:
        raise ValueError("lado deve ser 'esquerda', 'direita' ou 'ambos'.")

    return valores[-1]


#Cálculo de limites no infinito
def limite_no_infinito(f, sinal: int = 1, h_inicial: float = 1e6) -> float:
    if sinal not in (1, -1):
        raise ValueError("Sinal deve ser 1 (para +∞) ou −1 (para −∞).")

    valor_anterior = f(sinal * h_inicial)
    for k in range(1, 10):
        h = sinal * h_inicial * (10 ** k)
        valor_atual = f(h)
        if valor_absoluto(valor_atual - valor_anterior) < tolerancia_de_erro:
            return valor_atual
        valor_anterior = valor_atual

    return valor_anterior


#Derivação parcial de 1° ordem
def derivada_parcial(f, x: list, indice: int, h: float = 1e-5) -> float:
    
    x_mais = x[:]
    x_menos = x[:]
    
    x_mais[indice] += h
    x_menos[indice] -= h
    
    return (f(x_mais) - f(x_menos)) / (2.0 * h)


#Derivação parcial de 2° ordem
def derivada_parcial_segunda(f, x: list, indice_i: int, indice_j: int, h: float = 1e-4) -> float:
    if indice_i == indice_j:
        
        xp = x[:]
        xm = x[:]
        
        xp[indice_i] += h
        xm[indice_i] -= h
        
        return (f(xp) - 2 * f(x) + f(xm)) / (h * h)

    xpp = x[:]; xpm = x[:]; xmp = x[:]; xmm = x[:]
    
    xpp[indice_i] += h; xpp[indice_j] += h
    xpm[indice_i] += h; xpm[indice_j] -= h
    xmp[indice_i] -= h; xmp[indice_j] += h
    xmm[indice_i] -= h; xmm[indice_j] -= h
    
    return (f(xpp) - f(xpm) - f(xmp) + f(xmm)) / (4 * h * h)


#Zeros
#Definição das raízes
def encontrar_zeros(f, a: float, b: float, tolerancia: float = None) -> list:
    if tolerancia is None:
        tolerancia = tolerancia_de_erro

    N = 200
    dx = (b - a) / N
    zeros = []

    for i in range(N):
        xa = a + i * dx
        xb = xa + dx
        try:
            fa, fb = f(xa), f(xb)
        except Exception:
            continue

        if valor_absoluto(fa) < tolerancia:
            if not zeros or valor_absoluto(zeros[-1] - xa) > tolerancia * 10:
                zeros.append(xa)
        elif fa * fb < 0:
            z = bisseccao(f, xa, xb, tolerancia)
            if not zeros or valor_absoluto(zeros[-1] - z) > tolerancia * 10:
                zeros.append(z)

    return zeros


#Bissecção
def bisseccao(f, a: float, b: float, tolerancia: float) -> float:
    for _ in range(maximo_de_interacoes):
        meio = (a + b) / 2.0
        f_meio = f(meio)
        if valor_absoluto(f_meio) < tolerancia or (b - a) < tolerancia:
            return meio
        if f(a) * f_meio < 0:
            b = meio
        else:
            a = meio
    return (a + b) / 2.0


#Mudanças de comportamento
#Revelação dos extremos
def encontrar_extremos(f, a: float, b: float, h: float = 1e-5) -> dict:
    resultado = {"minimos": [], "maximos": [], "inflexoes": []}
    N = 500
    dx = (b - a) / N

    for i in range(1, N):
        x = a + i * dx
        try:
            d1 = derivada(f, x, h)
            d2_valor = (derivada(f, x + h * 10) - derivada(f, x - h * 10)) / (2 * h * 10)
        except Exception:
            continue

        if valor_absoluto(d1) < h * 100:
            if d2_val > h:
                resultado["minimos"].append((x, f(x)))
            elif d2_val < -h:
                resultado["maximos"].append((x, f(x)))
            else:
                resultado["inflexoes"].append((x, f(x)))

    return resultado


#Revelação dos pontos de inflexão
def pontos_de_inflexao(f, a: float, b: float, h: float = 1e-4, N: int = 500) -> list:
    def segunda_derivada(x):
        return (derivada(f, x + h, h) - derivada(f, x - h, h)) / (2.0 * h)

    dx = (b - a) / N
    inflexoes = []
    sinal_anterior = None

    for i in range(N + 1):
        x = a + i * dx
        try:
            d2 = segunda_derivada(x)
        except Exception:
            continue

        sinal_atual = 1 if d2 > tolerancia_de_erro else (-1 if d2 < -tolerancia_de_erro else 0)

        if sinal_anterior is not None and sinal_atual != 0 and sinal_anterior != 0:
            if sinal_atual != sinal_anterior:
                x_esquerda, x_direita = x - dx, x
                x_meio = x_esquerda
                for _ in range(maximo_de_interacoes):
                    x_meio = (x_esquerda + x_direita) / 2.0
                    d2_meio = segunda_derivada(x_meio)
                    if valor_absoluto(d2_meio) < tolerancia_de_erro or \
                       valor_absoluto(x_direita - x_esquerda) < tolerancia_de_erro:
                        break
                    if (d2_meio > 0) == (segunda_derivada(x_esquerda) > 0):
                        x_esquerda = x_meio
                    else:
                        x_direita = x_meio
                inflexoes.append((x_meio, f(x_meio)))

        if sinal_atual != 0:
            sinal_anterior = sinal_atual

    return inflexoes


#Verificação das propriedades
#Verificação da monotonia
def monotonia(f, a: float, b: float, N: int = 100) -> list:
    dx = (b - a) / N
    intervalos = []
    sinal_atual = None
    inicio = a

    for i in range(N):
        x = a + i * dx
        try:
            d = derivada(f, x)
        except Exception:
            continue

        if d > tolerancia_de_erro:
            s = "crescente"
        elif d < -tolerancia_de_erro:
            s = "decrescente"
        else:
            s = "constante"

        if s != sinal_atual:
            if sinal_atual is not None:
                intervalos.append((inicio, x, sinal_atual))
            inicio = x
            sinal_atual = s

    if sinal_atual:
        intervalos.append((inicio, b, sinal_atual))

    return intervalos


#Verificação se é continua
def eh_continua_em(f, x: float, h: float = 1e-6, tol: float = None) -> bool:
    if tol is None:
        tol = tolerancia_de_erro * 100

    try:
        valor_centro = f(x)
        limite_esquerda = f(x - h)
        limite_direita = f(x + h)
    except Exception:
        return False

    if valor_absoluto(limite_esquerda - limite_direita) > tol:
        return False
    if valor_absoluto(limite_esquerda - valor_centro) > tol:
        return False
    if valor_absoluto(limite_direita - valor_centro) > tol:
        return False

    return True


#Verificação se é par
def eh_par(f, intervalo: tuple = (-10, 10), n: int = 100, tol: float = None) -> bool:
    if tol is None:
        tol = tolerancia_de_erro * 1000

    a, b = intervalo
    dx = (b - a) / n
    for i in range(n + 1):
        x = a + i * dx
        if x == 0:
            continue
        try:
            if valor_absoluto(f(x) - f(-x)) > tol:
                return False
        except Exception:
            continue
    return True


#Verificação se é ímpar
def eh_impar(f, intervalo: tuple = (-10, 10), n: int = 100, tol: float = None) -> bool:
    if tol is None:
        tol = tolerancia_de_erro * 1000

    a, b = intervalo
    dx = (b - a) / n
    for i in range(n + 1):
        x = a + i * dx
        if x == 0:
            continue
        try:
            if valor_absoluto(f(x) + f(-x)) > tol:
                return False
        except Exception:
            continue
    return True


#Verificação de periodicidade
def eh_periodica(f, periodo_candidato: float, intervalo: tuple = (-10, 10), n: int = 100, tol: float = None) -> bool:
    if tol is None:
        tol = tolerancia_de_erro * 1000

    a, b = intervalo
    dx = (b - a) / n
    for i in range(n + 1):
        x = a + i * dx
        try:
            if valor_absoluto(f(x) - f(x + periodo_candidato)) > tol:
                return False
        except Exception:
            continue
    return True


#Teoria das funções
#Verificação se é injetora
def eh_injetora(f, intervalo: tuple, n: int = 200, tol: float = None) -> bool:
    if tol is None:
        tol = tolerancia_de_erro * 1000

    a, b = intervalo
    dx = (b - a) / n
    valores = []
    for i in range(n + 1):
        x = a + i * dx
        try:
            valores.append(f(x))
        except Exception:
            continue

    for i in range(len(valores)):
        for j in range(i + 1, len(valores)):
            if valor_absoluto(valores[i] - valores[j]) < tol:
                return False
    return True


#Verificação se é sobrejetora
def eh_sobrejetora(f, dominio: tuple, contradominio: tuple, n: int = 1000, tol: float = None) -> bool:
    if tol is None:
        tol = (contradominio[1] - contradominio[0]) / n * 2

    a, b = dominio
    dx = (b - a) / n
    valores_alcancados = []
    for i in range(n + 1):
        x = a + i * dx
        try:
            valores_alcancados.append(f(x))
        except Exception:
            continue

    valores_alcancados.sort()
    cd_inf, cd_sup = contradominio
    n_amostras_cd = 50
    passo_cd = (cd_sup - cd_inf) / n_amostras_cd

    for k in range(n_amostras_cd + 1):
        y = cd_inf + k * passo_cd
        encontrou = False
        for v in valores_alcancados:
            if valor_absoluto(v - y) < tol:
                encontrou = True
                break
        if not encontrou:
            return False
    return True


#Verificação se é bijetora
def eh_bijetora(f, dominio: tuple, contradominio: tuple, n: int = 1000, tol: float = None) -> bool:
    return eh_injetora(f, dominio, n, tol) and eh_sobrejetora(f, dominio, contradominio, n, tol)


#Imagem
def imagem_aproximada(f, dominio: tuple, n: int = 1000) -> tuple:
    a, b = dominio
    dx = (b - a) / n
    minimo = None
    maximo = None
    for i in range(n + 1):
        x = a + i * dx
        try:
            y = f(x)
        except Exception:
            continue
        if minimo is None or y < minimo:
            minimo = y
        if maximo is None or y > maximo:
            maximo = y
    return (minimo, maximo)


#Domínio
def dominio_aproximado(f, intervalo_teste: tuple = (-1000, 1000), n: int = 5000) -> list:
    a, b = intervalo_teste
    dx = (b - a) / n
    intervalos_validos = []
    inicio_valido = None

    for i in range(n + 1):
        x = a + i * dx
        valido = True
        try:
            valor = f(x)
            if valor != valor:
                valido = False
        except Exception:
            valido = False

        if valido and inicio_valido is None:
            inicio_valido = x
        elif not valido and inicio_valido is not None:
            intervalos_validos.append((inicio_valido, x - dx))
            inicio_valido = None

    if inicio_valido is not None:
        intervalos_validos.append((inicio_valido, b))

    return intervalos_validos


#Resolução de equações
#Ponto fixo
def ponto_fixo(f, x0: float, tolerancia: float = None, iteracoes: int = None) -> float:
    if tolerancia is None:
        tolerancia = tolerancia_de_erro
    if iteracoes is None:
        iteracoes = maximo_de_interacoes

    x = x0
    for _ in range(iteracoes):
        x_novo = f(x)
        if valor_absoluto(x_novo - x) < tolerancia:
            return x_novo
        x = x_novo

    raise ValueError(f"Ponto fixo não convergiu em {iteracoes} iterações.")


#Método de Newton-Raphson
def newton_zero(f, x0: float, tolerancia: float = None, iteracoes: int = None, h: float = 1e-6) -> float:
    if tolerancia is None:
        tolerancia = tolerancia_de_erro
    if iteracoes is None:
        iteracoes = maximo_de_interacoes

    x = x0
    for _ in range(iteracoes):
        fx = f(x)
        dfx = derivada(f, x, h)
        if valor_absoluto(dfx) < tol:
            raise ZeroDivisionError(f"Derivada ≈ 0 em x={x}: Newton não pode continuar.")
        x_novo = x - fx / dfx
        if valor_absoluto(x_novo - x) < tol:
            return x_novo
        x = x_novo

    raise ValueError(f"Newton não convergiu em {iteracoes} iterações.")


#Método da secante
def metodo_secante(f, x0: float, x1: float, tol: float = None, iteracoes: int = None) -> float:
    if tol is None:
        tol = tolerancia_de_erro
    if iteracoes is None:
        iteracoes = maximo_de_interacoes

    for _ in range(iteracoes):
        f0, f1 = f(x0), f(x1)
        if valor_absoluto(f1 - f0) < tol:
            raise ZeroDivisionError("Secante: f(x1) ≈ f(x0), divisão por zero.")
        x2 = x1 - f1 * (x1 - x0) / (f1 - f0)
        if valor_absoluto(x2 - x1) < tol:
            return x2
        x0, x1 = x1, x2

    raise ValueError(f"Secante não convergiu em {iteracoes} iterações.")


#Método da falsa posição
def metodo_falsa_posicao(f, a: float, b: float, tol: float = None, iteracoes: int = None) -> float:
    if tol is None:
        tol = tolerancia_de_erro
    if iteracoes is None:
        iteracoes = maximo_de_interacoes

    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError("regula_falsi requer f(a) e f(b) com sinais opostos.")

    for _ in range(iteracoes):
        c = (a * fb - b * fa) / (fb - fa)
        fc = f(c)
        if valor_absoluto(fc) < tol:
            return c
        if fa * fc < 0:
            b, fb = c, fc
        else:
            a, fa = c, fc

    return c

#Criar função xy
def criar_funcao_xy(expr: str):
    """Converte uma expressão em x e y numa função Python f(x, y)."""
    def f(x, y):
        return avaliar_expressao(expr, {"x": x, "y": y})
    f.expressao = expr
    return f

Overwriting calculo.py
